In [2]:
# version 7 2025.12.04 ~ 05

import torch
print(torch.cuda.is_available())      # True면 GPU 사용 가능
print(torch.cuda.get_device_name(0))  # GPU 이름 출력

# 학원 Workstation 환경: 32GB RAM, GTX 1660 Super, Ryzen 5 Pro 4650G

True
NVIDIA GeForce GTX 1660 SUPER


In [3]:
import pandas as pd
import numpy as np
import os
import json
import datetime
import gc
import re
import warnings
from tqdm import tqdm
from gensim.models import FastText
from sentence_transformers import SentenceTransformer

import glob
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from pycaret.regression import *
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


class MercariPyCaretAnalyzer7:
    """
    Mercari Price Suggestion Challenge를 위한 PyCaret 기반 머신러닝 파이프라인

    주요 기능: 데이터 전처리, 텍스트 벡터화(TF-IDF/FastText/BERT),
              피처 엔지니어링, 모델 학습/블렌딩, 성능 평가

    Parameters
    ----------
    data_dir : str, default="../data"
    images_dir : str, default="../images"
    results_dir : str, default="../results"
    model_dir : str, default="../models"
    """

    # __init__ ##############################
    def __init__(self, data_dir="../data", images_dir="../images",
                 results_dir="../results", model_dir="../models"):
        self.data_dir     = data_dir
        self.images_dir   = images_dir
        self.results_dir  = results_dir
        self.model_dir    = model_dir

        self.train        = None
        self.test         = None
        self.train_vectorized = None
        self.test_vectorized  = None

        self.best_model   = None
        self.setup_result = None
        self.metrics      = {}
        self.models       = {}   # 여러 모델을 메모리에 보관

        os.makedirs(self.images_dir,  exist_ok=True)
        os.makedirs(self.results_dir, exist_ok=True)
        os.makedirs(self.model_dir,   exist_ok=True)
    # eof -----------------------------------

    # _collapse_rare_values #################
    def _collapse_rare_values(self, col, top_k, rare_label="Other"):
        """카테고리 값 중 발생 빈도가 작은 값들을 rare_label로 통합"""
        combined = pd.concat([self.train[col], self.test[col]], axis=0)
        value_counts = combined.value_counts()
        top_values = set(value_counts.index[:top_k])

        self.train[col] = self.train[col].apply(
            lambda x: x if x in top_values else rare_label
        )
        self.test[col] = self.test[col].apply(
            lambda x: x if x in top_values else rare_label
        )

        del combined, value_counts
        gc.collect()
    # eof -----------------------------------

    # _simple_normalize #####################
    def _simple_normalize(self, text: str) -> str:
        """간단 텍스트 정규화 (현재는 사용 X, 필요하면 확장용)"""
        text = str(text).lower()
        text = re.sub(r"[_\-\./]", " ", text)
        text = re.sub(r"\d+", " num ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text
    # eof -----------------------------------

    # _stratified_sample ####################
    def _stratified_sample(self, frac=0.35, bins=10):
        """price를 구간으로 나눠 층화 샘플링 (언더샘플링)"""
        self.train["price_bin"] = pd.qcut(
            self.train["price"], q=bins, duplicates="drop"
        )
        sampled = self.train.groupby("price_bin", group_keys=False).apply(
            lambda x: x.sample(frac=frac, random_state=23)
        )
        self.train = sampled.drop(columns=["price_bin"]).reset_index(drop=True)
        gc.collect()
        print(f"⚠️ Stratified undersampling 적용: train {self.train.shape}")
    # eof -----------------------------------

    # load_data #############################
    def load_data(self, train_file="train.tsv", test_file="test.tsv",
                  sep="\t", undersample_frac=0.35):
        """데이터 로딩 + 기본 전처리 (카테고리 분리, 결측치 처리, 피처 생성)"""
        print("📂 데이터 로딩 시작...")
        self.train = pd.read_csv(os.path.join(self.data_dir, train_file), sep=sep)
        self.test  = pd.read_csv(os.path.join(self.data_dir, test_file),  sep=sep)

        # price > 0, log1p 변환
        self.train = self.train[self.train["price"] > 0].dropna(subset=["price"])
        self.train["price"] = np.log1p(self.train["price"])

        # 층화 언더샘플링
        if undersample_frac:
            self._stratified_sample(frac=undersample_frac)

        # category_name 분해 + 기본 텍스트/브랜드 처리
        for df in [self.train, self.test]:
            df["main_cat"], df["sub_cat"], df["sub_sub_cat"] = zip(
                *df["category_name"].apply(
                    lambda x: (x.split("/") if isinstance(x, str) and "/" in x
                               else ["missing"] * 3)
                )
            )
            df["brand_name"]       = df["brand_name"].fillna("Unknown").astype(str)
            df["item_description"] = df["item_description"].fillna("No description").astype(str)
            df["name"]             = df["name"].fillna("No name").astype(str)
            df.drop(columns=["category_name"], inplace=True)

        # 희귀 카테고리 통합
        print("🔄 희귀값 통합 중...")
        self._collapse_rare_values("brand_name", 5000, "Other_brand")
        self._collapse_rare_values("main_cat",   1000, "Other_main")
        self._collapse_rare_values("sub_cat",    1000, "Other_sub")
        self._collapse_rare_values("sub_sub_cat", 1000, "Other_sub_sub")

        # 간단 길이/매칭 피처
        print("📏 피처 생성 중...")
        for df in [self.train, self.test]:
            df["name_len_char"]  = df["name"].str.len()
            df["name_len_word"]  = df["name"].str.split().str.len()
            df["desc_len_char"]  = df["item_description"].str.len()
            df["desc_len_word"]  = df["item_description"].str.split().str.len()
            df["has_brand_in_name"] = df.apply(
                lambda r: 1 if r["brand_name"].lower() in r["name"].lower() else 0,
                axis=1,
            )
            df["has_brand_in_desc"] = df.apply(
                lambda r: 1 if r["brand_name"].lower() in r["item_description"].lower()
                else 0,
                axis=1,
            )
            df["shipping"]          = df["shipping"].astype("category")
            df["item_condition_id"] = df["item_condition_id"].astype("category")

        gc.collect()
        print(f"✅ 데이터 로드 완료: train {self.train.shape}, test {self.test.shape}")
    # eof -----------------------------------

    # vectorize_text_tfidf ##################
    def vectorize_text_tfidf(
        self, max_features_name=15000, max_features_desc=20000,
        n_components=150
    ):
        """TF-IDF + SVD 벡터화"""
        print("🔍 TF-IDF 벡터화 시작...")

        # name
        vec_name = TfidfVectorizer(
            max_features=max_features_name,
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.95,
            sublinear_tf=True,
            dtype=np.float32,
        )
        name_train = vec_name.fit_transform(self.train["name"])
        name_test  = vec_name.transform(self.test["name"])

        svd_name = TruncatedSVD(
            n_components=min(n_components, name_train.shape[1] - 1),
            random_state=23,
        )
        name_train_svd = svd_name.fit_transform(name_train)
        name_test_svd  = svd_name.transform(name_test)
        print(f"   - name SVD: {name_train_svd.shape}, 설명력={svd_name.explained_variance_ratio_.sum():.2%}")

        del name_train, name_test, vec_name
        gc.collect()

        # description
        vec_desc = TfidfVectorizer(
            max_features=max_features_desc,
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.95,
            sublinear_tf=True,
            dtype=np.float32,
        )
        desc_train = vec_desc.fit_transform(self.train["item_description"])
        desc_test  = vec_desc.transform(self.test["item_description"])

        svd_desc = TruncatedSVD(
            n_components=min(n_components, desc_train.shape[1] - 1),
            random_state=23,
        )
        desc_train_svd = svd_desc.fit_transform(desc_train)
        desc_test_svd  = svd_desc.transform(desc_test)
        print(f"   - desc SVD: {desc_train_svd.shape}, 설명력={svd_desc.explained_variance_ratio_.sum():.2%}")

        del desc_train, desc_test, vec_desc, svd_desc
        gc.collect()

        # 합치기
        train_vec = np.hstack([name_train_svd, desc_train_svd]).astype(np.float32)
        test_vec  = np.hstack([name_test_svd,  desc_test_svd]).astype(np.float32)

        del name_train_svd, name_test_svd, desc_train_svd, desc_test_svd
        gc.collect()

        self.train_vectorized = pd.DataFrame(
            train_vec,
            columns=[f"name_{i}" for i in range(n_components)]
                    + [f"desc_{i}" for i in range(n_components)],
        )
        self.test_vectorized = pd.DataFrame(
            test_vec,
            columns=[f"name_{i}" for i in range(n_components)]
                    + [f"desc_{i}" for i in range(n_components)],
        )

        self._add_categorical_numeric_features()
        print(f"✅ TF-IDF 완료: train {self.train_vectorized.shape}, test {self.test_vectorized.shape}")
    # eof -----------------------------------

    # vectorize_text_fasttext ###############
    def vectorize_text_fasttext(
        self,
        text_columns=["name", "item_description"],
        fasttext_size=100,
        fasttext_window=5,
        fasttext_min_count=2,
        n_components=None,
    ):
        """FastText 임베딩 + (선택) PCA 축소"""
        print("🔍 FastText 벡터화 시작...")

        sentences = []
        for col in text_columns:
            self.train[col] = self.train[col].fillna("").astype(str)
            self.test[col]  = self.test[col].fillna("").astype(str)
            sentences += [str(x).split()
                          for x in pd.concat([self.train[col], self.test[col]])]

        print("   - FastText 학습 중...")
        ft_model = FastText(
            sentences,
            vector_size=fasttext_size,
            window=fasttext_window,
            min_count=fasttext_min_count,
            sg=1,
            workers=4,
        )

        def get_vector(text):
            words = str(text).split()
            vectors = [ft_model.wv[w] for w in words if w in ft_model.wv]
            return np.mean(vectors, axis=0) if vectors else np.zeros(fasttext_size)

        train_features, test_features = [], []
        for col in tqdm(text_columns, desc="벡터 생성"):
            train_features.append(np.vstack(self.train[col].apply(get_vector)))
            test_features.append(np.vstack(self.test[col].apply(get_vector)))

        train_vec = np.hstack(train_features).astype(np.float32)
        test_vec  = np.hstack(test_features).astype(np.float32)

        # 선택: PCA 축소
        if n_components:
            from sklearn.decomposition import PCA
            pca = PCA(n_components=n_components, random_state=42)
            train_vec = pca.fit_transform(train_vec)
            test_vec  = pca.transform(test_vec)

        # 컬럼 이름 구성
        if n_components:
            dim_per_col = n_components // len(text_columns)
        else:
            dim_per_col = train_vec.shape[1] // len(text_columns)

        self.train_vectorized = pd.DataFrame(
            train_vec,
            columns=[
                f"{col}_ft_{i}"
                for col in text_columns
                for i in range(dim_per_col)
            ],
        )
        self.test_vectorized = pd.DataFrame(
            test_vec,
            columns=[
                f"{col}_ft_{i}"
                for col in text_columns
                for i in range(dim_per_col)
            ],
        )

        self._add_categorical_numeric_features()
        print(f"✅ FastText 완료: train {self.train_vectorized.shape}, test {self.test_vectorized.shape}")
    # eof -----------------------------------

    # vectorize_text_bert ###################
    def vectorize_text_bert(
        self,
        text_columns=["name", "item_description"],
        bert_model_name="all-MiniLM-L6-v2",
    ):
        """Sentence-BERT 임베딩 (GPU 사용)"""
        print(f"🔍 BERT 시작 (모델={bert_model_name})...")
        bert_model = SentenceTransformer(bert_model_name)

        train_features, test_features = [], []
        for col in text_columns:
            self.train[col] = self.train[col].fillna("").astype(str)
            self.test[col]  = self.test[col].fillna("").astype(str)
            print(f"   - {col} 인코딩 중...")
            train_features.append(
                bert_model.encode(
                    self.train[col].tolist(),
                    show_progress_bar=True,
                    batch_size=32,
                )
            )
            test_features.append(
                bert_model.encode(
                    self.test[col].tolist(),
                    show_progress_bar=True,
                    batch_size=32,
                )
            )

        train_vec = np.hstack(train_features).astype(np.float32)
        test_vec  = np.hstack(test_features).astype(np.float32)
        emb_dim   = bert_model.get_sentence_embedding_dimension()

        self.train_vectorized = pd.DataFrame(
            train_vec,
            columns=[
                f"{col}_bert_{i}"
                for col in text_columns
                for i in range(emb_dim)
            ],
        )
        self.test_vectorized = pd.DataFrame(
            test_vec,
            columns=[
                f"{col}_bert_{i}"
                for col in text_columns
                for i in range(emb_dim)
            ],
        )

        self._add_categorical_numeric_features()
        print(f"✅ BERT 완료: train {self.train_vectorized.shape}, test {self.test_vectorized.shape}")
    # eof -----------------------------------

    # _add_categorical_numeric_features #####
    def _add_categorical_numeric_features(self):
        """벡터라이즈된 데이터프레임에 범주형/수치형 피처 추가"""
        categorical_features = [
            "main_cat",
            "sub_cat",
            "sub_sub_cat",
            "brand_name",
            "item_condition_id",
            "shipping",
        ]
        numeric_features = [
            "name_len_char",
            "name_len_word",
            "desc_len_char",
            "desc_len_word",
            "has_brand_in_name",
            "has_brand_in_desc",
        ]

        print("📌 범주형/수치형 피처 추가 중...")
        for col in categorical_features + numeric_features:
            if col in self.train.columns:
                self.train_vectorized[col] = self.train[col].reset_index(drop=True)
                self.test_vectorized[col]  = self.test[col].reset_index(drop=True)

        print(
            f"   - 범주형:{len(categorical_features)}개, 수치형:{len(numeric_features)}개, "
            f"총:{self.train_vectorized.shape[1]}개"
        )
    # eof -----------------------------------

    # vectorize_text ########################
    def vectorize_text(self, method="tfidf", **kwargs):
        """
        텍스트 벡터화 통합 인터페이스
        method: 'tfidf', 'fasttext', 'bert'
        """
        if self.load_vectorized(method):
            return

        if method == "tfidf":
            allowed = {
                "max_features_name",
                "max_features_desc",
                "n_components",
                "text_columns",
            }
            args = {k: v for k, v in kwargs.items() if k in allowed}
            self.vectorize_text_tfidf(**args)

        elif method == "fasttext":
            allowed = {
                "text_columns",
                "fasttext_size",
                "fasttext_window",
                "fasttext_min_count",
                "n_components",
            }
            args = {k: v for k, v in kwargs.items() if k in allowed}
            self.vectorize_text_fasttext(**args)

        elif method == "bert":
            allowed = {"text_columns", "bert_model_name"}
            args = {k: v for k, v in kwargs.items() if k in allowed}
            self.vectorize_text_bert(**args)

        else:
            raise ValueError("method must be one of ['tfidf', 'fasttext', 'bert']")

        self.save_vectorized(method)
    # eof -----------------------------------

    # setup_pycaret #########################
    def setup_pycaret(self, session_id=23, fold=3, use_gpu=True):
        """PyCaret 환경 설정"""
        print("🔧 PyCaret setup 시작...")

        categorical_cols = [
            "main_cat",
            "sub_cat",
            "sub_sub_cat",
            "brand_name",
            "item_condition_id",
            "shipping",
        ]
        existing_categorical = [
            col for col in categorical_cols
            if col in self.train_vectorized.columns
        ]
        print(
            f"   - 범주형:{len(existing_categorical)}개, "
            f"전체:{self.train_vectorized.shape[1]}개"
        )

        self.setup_result = setup(
            data=self.train_vectorized.assign(
                price=self.train["price"].reset_index(drop=True)
            ),
            target="price",
            session_id=session_id,
            categorical_features=(
                existing_categorical if existing_categorical else None
            ),
            normalize=True,
            transformation=False,
            fold_strategy="kfold",
            fold=fold,
            use_gpu=use_gpu,
            n_jobs=4,
            verbose=True,
            html=False,
        )
        gc.collect()
        print("✅ PyCaret setup 완료")
    # eof -----------------------------------

    # find_and_blend_models #################
    def find_and_blend_models(
        self, top_n=3, sort_metric="R2", use_kaggle_winners=True
    ):
        """상위권에서 자주 쓰는 모델들로 앙상블 구성"""
        if not self.setup_result:
            raise ValueError("먼저 setup_pycaret()를 실행하세요.")

        if use_kaggle_winners:
            print("🏆 Kaggle 상위권 5개 모델 학습...")
            model_names = ["lightgbm", "ridge", "catboost", "xgboost", "et"]
            top_models = [
                create_model(name, verbose=False)
                for name in tqdm(model_names, desc="모델 학습")
            ]
            print("✅ 5개 모델 학습 완료")
        else:
            print(f"🔍 compare_models로 상위 {top_n}개 선택...")
            top_models = compare_models(
                n_select=top_n, sort=sort_metric, turbo=True, verbose=True
            )
            if not isinstance(top_models, list):
                top_models = [top_models]

        print("🎯 선정 모델:")
        for i, m in enumerate(top_models, 1):
            print(f"   {i}. {str(m).split('(')[0]}")

        print(f"\n🔀 {len(top_models)}개 모델 블렌딩...")
        blended = blend_models(
            estimator_list=top_models,
            optimize=sort_metric,
            choose_better=True,
            verbose=True,
        )
        self.best_model = blended
        gc.collect()
        print(f"🏆 Blended model 완료 (기준={sort_metric})")

        model_tag = f"blended_{sort_metric}"
        self.save_best_model(model_name=model_tag)

        return self.best_model
    # eof -----------------------------------

    # tune_best_model #######################
    def tune_best_model(self, n_iter=50, optimize_metric="R2"):
        """현재 best_model을 기반으로 하이퍼파라미터 튜닝"""
        if not self.best_model:
            raise ValueError("먼저 find_and_blend_models()를 실행하세요.")

        print(f"⚙️ 튜닝 시작 (n_iter={n_iter})...")
        tuned = tune_model(
            self.best_model,
            optimize=optimize_metric,
            n_iter=n_iter,
            search_library="optuna",
            search_algorithm="tpe",
        )
        self.best_model = tuned
        print("✅ 튜닝 완료!")

        model_tag = f"tuned_{optimize_metric}"
        self.save_best_model(model_name=model_tag)

        return tuned
    # eof -----------------------------------

    # save_metrics ##########################
    def save_metrics(self, model_name=None):
        """train 전체에 대해 성능 지표 계산 + JSON 저장 (price 스케일 기준)"""
        if not self.best_model:
            raise ValueError("모델이 없습니다.")

        print("📊 성능 평가 중 (train 전체)...")
        pred_df = predict_model(
            self.best_model, data=self.train_vectorized.copy()
        )

        y_true = np.expm1(self.train["price"].values)
        y_pred = np.expm1(pred_df["prediction_label"].values)

        self.metrics = {
            "R2":   round(r2_score(y_true, y_pred), 4),
            "RMSE": round(mean_squared_error(y_true, y_pred, squared=False), 4),
            "MAE":  round(mean_absolute_error(y_true, y_pred), 4),
        }

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        model_name = model_name or str(self.best_model).split("(")[0]
        file_path = os.path.join(
            self.results_dir, f"{model_name}_metrics_{timestamp}.json"
        )
        with open(file_path, "w") as f:
            json.dump(self.metrics, f, indent=4)

        print(f"💾 Metrics 저장: {file_path}")
        print(
            f"   - R²={self.metrics['R2']}, "
            f"RMSE=${self.metrics['RMSE']:.2f}, "
            f"MAE=${self.metrics['MAE']:.2f}"
        )
    # eof -----------------------------------

    # predict_test ##########################
    def predict_test(self, submission_file="submission.csv"):
        """test.tsv에 대해 예측 + submission 저장"""
        if not self.best_model:
            raise ValueError("먼저 find_and_blend_models()를 실행하세요.")

        print("📦 Test 예측 시작...")
        predictions = predict_model(
            self.best_model, data=self.test_vectorized.copy()
        )
        price_pred = np.expm1(predictions["prediction_label"].values)

        submission = pd.DataFrame(
            {"test_id": self.test["test_id"], "price": price_pred}
        )
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        path = os.path.join(self.results_dir, f"{timestamp}_{submission_file}")
        submission.to_csv(path, index=False)

        print(f"💾 Submission 저장: {path}")
        print(
            f"   - 가격 범위: ${price_pred.min():.2f} ~ "
            f"${price_pred.max():.2f}, 평균: ${price_pred.mean():.2f}"
        )
        return submission
    # eof -----------------------------------

    # save_vectorized #######################
    def save_vectorized(self, method="tfidf"):
        """벡터화 결과 (train/test)를 model_dir에 피클로 저장"""
        train_path = os.path.join(
            self.model_dir, f"vectorized_{method}_train.pkl"
        )
        test_path = os.path.join(
            self.model_dir, f"vectorized_{method}_test.pkl"
        )
        self.train_vectorized.to_pickle(train_path)
        self.test_vectorized.to_pickle(test_path)
        print(f"💾 {method} 벡터화 결과 저장 완료")
    # eof -----------------------------------

    # load_vectorized #######################
    def load_vectorized(self, method="tfidf"):
        """이미 저장된 벡터화 결과가 있으면 불러오기"""
        train_path = os.path.join(
            self.model_dir, f"vectorized_{method}_train.pkl"
        )
        test_path = os.path.join(
            self.model_dir, f"vectorized_{method}_test.pkl"
        )
        if os.path.exists(train_path) and os.path.exists(test_path):
            self.train_vectorized = pd.read_pickle(train_path)
            self.test_vectorized  = pd.read_pickle(test_path)
            print(
                f"📂 {method} 벡터화 결과 로드: "
                f"train {self.train_vectorized.shape}, "
                f"test {self.test_vectorized.shape}"
            )
            return True
        print(f"⚠️ {method} 벡터화 결과 없음")
        return False
    # eof -----------------------------------

    # save_best_model #######################
    def save_best_model(self, model_name=None):
        """학습된 best_model을 ../models에 저장 (버전 + latest)"""
        if not self.best_model:
            raise ValueError("저장할 모델이 없습니다. 먼저 find_and_blend_models() 실행 필요.")

        os.makedirs(self.model_dir, exist_ok=True)

        base_name = model_name or str(self.best_model).split("(")[0]
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

        versioned_path = os.path.join(
            self.model_dir, f"{base_name}_{timestamp}"
        )
        save_model(self.best_model, versioned_path)

        latest_path = os.path.join(self.model_dir, f"{base_name}_latest")
        save_model(self.best_model, latest_path)

        print(
            f"💾 모델 저장 완료: {versioned_path}.pkl (버전), "
            f"{latest_path}.pkl (최신)"
        )
    # eof -----------------------------------

    # load_saved_model ######################
    def load_saved_model(self, model_name, latest=True):
        """../models에서 저장된 모델 로드"""
        suffix = "_latest" if latest else ""
        path = os.path.join(self.model_dir, f"{model_name}{suffix}")
        self.best_model = load_model(path)
        print(f"📂 모델 불러오기 완료: {path}.pkl")
        return self.best_model
    # eof -----------------------------------

    # evaluate_and_save_models ##############
    def evaluate_and_save_models(self, valid_data):
        """
        ../models에 저장된 모든 모델을 불러와 성능 평가 + JSON 저장 + 시각화.
        valid_data: price(로그 스케일) 컬럼을 포함한 DataFrame
                    (예: self.train_vectorized.assign(price=self.train['price']))
        """
        model_files = glob.glob(os.path.join(self.model_dir, "*.pkl"))
        if not model_files:
            raise ValueError("📂 models 디렉토리에 저장된 모델이 없습니다.")

        results = []
        for mf in model_files:
            model_name = os.path.basename(mf).replace(".pkl", "")
            model = load_model(mf)
            preds = predict_model(model, data=valid_data.copy())

            y_true = np.expm1(valid_data["price"].values)
            y_pred = np.expm1(preds["prediction_label"].values)

            rmse = mean_squared_error(y_true, y_pred, squared=False)
            mae  = mean_absolute_error(y_true, y_pred)
            r2   = r2_score(y_true, y_pred)

            metrics = {
                "R2": round(r2, 4),
                "RMSE": round(rmse, 4),
                "MAE": round(mae, 4),
            }
            results.append({"model": model_name, **metrics})

            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            file_path = os.path.join(
                self.results_dir, f"{model_name}_metrics_{timestamp}.json"
            )
            with open(file_path, "w") as f:
                json.dump(metrics, f, indent=4)
            print(f"💾 Metrics 저장: {file_path}")

        df_results = pd.DataFrame(results)
        print("📊 모델 성능 비교 결과:")
        print(df_results)

        # 시각화
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        df_results.plot(
            x="model", y="RMSE", kind="bar",
            ax=axes[0], color="skyblue", legend=False
        )
        axes[0].set_title("RMSE 비교")
        axes[0].set_ylabel("RMSE")

        df_results.plot(
            x="model", y="MAE", kind="bar",
            ax=axes[1], color="orange", legend=False
        )
        axes[1].set_title("MAE 비교")
        axes[1].set_ylabel("MAE")

        df_results.plot(
            x="model", y="R2", kind="bar",
            ax=axes[2], color="green", legend=False
        )
        axes[2].set_title("R² 비교")
        axes[2].set_ylabel("R² Score")

        plt.suptitle("저장된 모델 성능 비교", fontsize=16)
        plt.tight_layout()
        plt.show()

        return df_results
    # eof -----------------------------------

# End of class ###########################

In [4]:
# ============================================
# 1) 데이터 로드 + fastText 벡터화
# ============================================
analyzer = MercariPyCaretAnalyzer7()

In [5]:
# (1) 데이터 로드
analyzer.load_data(undersample_frac=0.35)  # 필요하면 0.5, 0.7 등으로 조정 가능

📂 데이터 로딩 시작...
⚠️ Stratified undersampling 적용: train (518582, 8)
🔄 희귀값 통합 중...
📏 피처 생성 중...
✅ 데이터 로드 완료: train (518582, 16), test (693359, 15)


In [6]:
# (2) fastText 벡터화 (이미 저장된 벡터가 있으면 load_vectorized를 통해 재사용)
analyzer.vectorize_text(method="fasttext")

📂 fasttext 벡터화 결과 로드: train (518582, 212), test (693359, 212)


In [7]:
# ============================================
# 2) Train / Valid 분리 (현실적인 성능 확인용)
# ============================================
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

n_samples = len(analyzer.train_vectorized)

# price를 구간으로 나눠서 층화(straified) 분리 → 비싼/싼 물건 비율 유지
price_bins = pd.qcut(
    analyzer.train["price"],
    q=10,
    duplicates="drop"
)

train_idx, valid_idx = train_test_split(
    np.arange(n_samples),
    test_size=0.1,       # 10%를 검증 셋으로 사용
    random_state=23,
    stratify=price_bins
)

X_train = analyzer.train_vectorized.iloc[train_idx].reset_index(drop=True)
X_valid = analyzer.train_vectorized.iloc[valid_idx].reset_index(drop=True)
y_train = analyzer.train["price"].iloc[train_idx].reset_index(drop=True)
y_valid = analyzer.train["price"].iloc[valid_idx].reset_index(drop=True)

print("Train shape :", X_train.shape)
print("Valid shape :", X_valid.shape)

Train shape : (466723, 212)
Valid shape : (51859, 212)


In [8]:
# ============================================
# 3) PyCaret setup (train split만 사용)
# ============================================
from pycaret.regression import setup, create_model, tune_model, finalize_model, predict_model

reg_exp = setup(
    data=X_train.assign(price=y_train),   # feature + 타깃 결합
    target="price",
    session_id=23,
    normalize=True,
    transformation=False,
    fold_strategy="kfold",
    fold=3,
    use_gpu=True,        # GPU 사용 (CatBoost, LightGBM 등)
    n_jobs=4,
    html=False,
    verbose=True
)

[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 2, number of used features: 0
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce GTX 1660 SUPER, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 16 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Warning] GPU acceleration is disabled because no non-trivial dense features can be found
[LightGBM] [Info] Start training from score 0.500000
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no

In [9]:
# ============================================
# 4) CatBoost 단일 모델 생성 + 튜닝
# ============================================
# (1) 기본 CatBoost 모델 생성
cat = create_model("catboost")

         MAE     MSE    RMSE      R2   RMSLE    MAPE
Fold                                                
0     0.3770  0.2478  0.4978  0.5538  0.1241  0.1337
1     0.3746  0.2449  0.4949  0.5594  0.1237  0.1331
2     0.3757  0.2461  0.4961  0.5598  0.1238  0.1334
Mean  0.3758  0.2463  0.4963  0.5577  0.1239  0.1334
Std   0.0010  0.0012  0.0012  0.0027  0.0002  0.0003


In [11]:
# (2) Optuna(TPE) 기반 튜닝 - 너무 오래 안 걸리게 n_iter를 적당히
tuned_cat = tune_model(
    cat,
    optimize="R2",
    n_iter=20,                 # 시간이 괜찮으면 30~40까지 올려도 됨
    search_library="optuna",
    search_algorithm="tpe"
)

Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).
         MAE     MSE    RMSE      R2   RMSLE    MAPE
Fold                                                
0     0.3837  0.2563  0.5063  0.5385  0.1262  0.1360
1     0.3813  0.2533  0.5032  0.5444  0.1257  0.1354
2     0.3824  0.2544  0.5044  0.5449  0.1259  0.1357
Mean  0.3824  0.2547  0.5046  0.5426  0.1259  0.1357
Std   0.0010  0.0013  0.0013  0.0029  0.0002  0.0003


In [12]:
# (3) finalize_model: 현재 setup의 전체 train 데이터(X_train)로 다시 학습
final_cat = finalize_model(tuned_cat)

In [13]:
# analyzer 안에도 best_model로 넣어두기 (뒤에서 predict_test에 사용)
analyzer.best_model = final_cat

In [14]:
# ============================================
# 5) Train / Valid에서 성능 평가 (로그 되돌린 스케일)
# ============================================
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

def eval_on_dataset(model, X, y_log, name="set"):
    """로그 스케일 y를 받아서, 원 스케일로 복원 후 성능 계산."""
    preds = predict_model(model, data=X.copy())
    y_true = np.expm1(y_log.values)
    y_pred = np.expm1(preds["prediction_label"].values)

    r2   = r2_score(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mae  = mean_absolute_error(y_true, y_pred)

    print(f"\n[{name}] 성능:")
    print(f" - R2   : {r2:.4f}")
    print(f" - RMSE : {rmse:.4f}")
    print(f" - MAE  : {mae:.4f}")
    return r2, rmse, mae

r2_tr, rmse_tr, mae_tr = eval_on_dataset(final_cat, X_train, y_train, name="Train")
r2_va, rmse_va, mae_va = eval_on_dataset(final_cat, X_valid, y_valid, name="Valid")


[Train] 성능:
 - R2   : 0.4103
 - RMSE : 29.9037
 - MAE  : 10.7908

[Valid] 성능:
 - R2   : 0.4274
 - RMSE : 27.3198
 - MAE  : 10.8269


In [15]:
# ============================================
# 6) PyCaret + CatBoost 모델 저장 (버전 + latest 둘 다)
# ============================================
analyzer.save_best_model(model_name="catboost_fasttext")

Transformation Pipeline and Model Successfully Saved
Transformation Pipeline and Model Successfully Saved
💾 모델 저장 완료: ../models\catboost_fasttext_20251205_183013.pkl (버전), ../models\catboost_fasttext_latest.pkl (최신)


In [17]:
# ============================================
# 7) 전체 train 데이터로 다시 학습해서 최종 제출 만들고 싶다면 (선택)
#    → 시간이 괜찮으면 이 블록을 실행
# ============================================
# 전체 train으로 다시 setup & 학습
reg_exp_full = setup(
    data=analyzer.train_vectorized.assign(price=analyzer.train["price"].reset_index(drop=True)),
    target="price",
    session_id=23,
    normalize=True,
    transformation=False,
    fold_strategy="kfold",
    fold=3,
    use_gpu=True,
    n_jobs=4,
    html=False,
    verbose=True
)


[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 2, number of used features: 0
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce GTX 1660 SUPER, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 16 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Warning] GPU acceleration is disabled because no non-trivial dense features can be found
[LightGBM] [Info] Start training from score 0.500000
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no

In [18]:

cat_full      = create_model("catboost")
tuned_cat_full = tune_model(
    cat_full,
    optimize="R2",
    n_iter=20,
    search_library="optuna",
    search_algorithm="tpe"
)


final_cat_full = finalize_model(tuned_cat_full)

         MAE     MSE    RMSE      R2   RMSLE    MAPE
Fold                                                
0     0.3760  0.2460  0.4959  0.5574  0.1240  0.1337
1     0.3724  0.2415  0.4915  0.5674  0.1228  0.1322
2     0.3753  0.2460  0.4960  0.5603  0.1236  0.1329
Mean  0.3746  0.2445  0.4945  0.5617  0.1234  0.1329
Std   0.0016  0.0021  0.0021  0.0042  0.0005  0.0006


Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).
         MAE     MSE    RMSE      R2   RMSLE    MAPE
Fold                                                
0     0.3823  0.2543  0.5043  0.5424  0.1260  0.1359
1     0.3792  0.2496  0.4996  0.5529  0.1248  0.1346
2     0.3826  0.2551  0.5050  0.5441  0.1258  0.1354
Mean  0.3813  0.2530  0.5030  0.5464  0.1255  0.1353
Std   0.0015  0.0024  0.0024  0.0046  0.0005  0.0006


In [19]:
analyzer.best_model = final_cat_full
analyzer.save_best_model(model_name="catboost_fasttext_full")

Transformation Pipeline and Model Successfully Saved
Transformation Pipeline and Model Successfully Saved
💾 모델 저장 완료: ../models\catboost_fasttext_full_20251205_184847.pkl (버전), ../models\catboost_fasttext_full_latest.pkl (최신)


In [20]:
# ============================================
# 8) Test 예측 + 제출 파일 생성
# ============================================
submission = analyzer.predict_test(
    submission_file="submission_pycaret_cat_fasttext.csv"
)

print("\n✅ PyCaret + fastText + CatBoost 파이프라인 완료!")

📦 Test 예측 시작...
💾 Submission 저장: ../results\20251205_184907_submission_pycaret_cat_fasttext.csv
   - 가격 범위: $2.03 ~ $694.62, 평균: $22.45

✅ PyCaret + fastText + CatBoost 파이프라인 완료!


In [21]:
# ============================================
# 5-1) Train / Valid 성능을 JSON으로 저장
# ============================================
import json, os, datetime

metrics_summary = {
    "model": "CatBoost_fasttext_tuned",
    "train": {
        "R2":   float(r2_tr),
        "RMSE": float(rmse_tr),
        "MAE":  float(mae_tr),
    },
    "valid": {
        "R2":   float(r2_va),
        "RMSE": float(rmse_va),
        "MAE":  float(mae_va),
    },
    "note": "PyCaret CatBoost + fastText, 90/10 stratified split, target=log1p(price)"
}

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
json_path = os.path.join(
    analyzer.results_dir,
    f"catboost_fasttext_split_metrics_{timestamp}.json"
)

with open(json_path, "w") as f:
    json.dump(metrics_summary, f, indent=4)

print(f"💾 Train/Valid metrics JSON 저장 완료: {json_path}")
print(metrics_summary)


💾 Train/Valid metrics JSON 저장 완료: ../results\catboost_fasttext_split_metrics_20251205_185217.json
{'model': 'CatBoost_fasttext_tuned', 'train': {'R2': 0.4103397398371329, 'RMSE': 29.903678337201146, 'MAE': 10.790846980329043}, 'valid': {'R2': 0.4273812025159204, 'RMSE': 27.31983872519525, 'MAE': 10.826942411287746}, 'note': 'PyCaret CatBoost + fastText, 90/10 stratified split, target=log1p(price)'}


In [22]:
# ============================================
# (선택) full train 모델을 Valid 셋에서 평가 + JSON 저장
# ============================================
r2_va_full, rmse_va_full, mae_va_full = eval_on_dataset(
    final_cat_full, X_valid, y_valid, name="Valid_full_model"
)

metrics_full = {
    "model": "CatBoost_fasttext_fullTrain",
    "valid": {
        "R2":   float(r2_va_full),
        "RMSE": float(rmse_va_full),
        "MAE":  float(mae_va_full),
    },
    "note": "full train으로 재학습한 CatBoost 모델을 같은 Valid 셋에 평가한 결과"
}

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
json_path_full = os.path.join(
    analyzer.results_dir,
    f"catboost_fasttext_full_valid_metrics_{timestamp}.json"
)

with open(json_path_full, "w") as f:
    json.dump(metrics_full, f, indent=4)

print(f"💾 Full-train 모델 Valid metrics JSON 저장 완료: {json_path_full}")
print(metrics_full)


[Valid_full_model] 성능:
 - R2   : 0.4555
 - RMSE : 26.6414
 - MAE  : 10.5653
💾 Full-train 모델 Valid metrics JSON 저장 완료: ../results\catboost_fasttext_full_valid_metrics_20251205_185318.json
{'model': 'CatBoost_fasttext_fullTrain', 'valid': {'R2': 0.45546801522549096, 'RMSE': 26.641399082234837, 'MAE': 10.565305897131923}, 'note': 'full train으로 재학습한 CatBoost 모델을 같은 Valid 셋에 평가한 결과'}
